# Omnilingual ASR vs Whisper - Urdu + GEC (FLEURS + UrduSpeech) - **v3**

Same experiment as `Qwen_vs_Whisper_GEC_comparison_v2.ipynb`, with three changes
(every edit in the code cells below is marked `CHANGED (v3)`):

1. **Qwen3-ASR is replaced by Meta's Omnilingual ASR** (`omniASR_LLM_1B_v2`). Urdu
   (`urd_Arab`) is natively supported and open-weights, so the forced-Hindi detour is gone.
2. **The Devanagari->Urdu transliteration stage is removed entirely** - Aksharamukha is no
   longer installed, `deva_to_urdu` is deleted, and the `qwen_hindi_deva` column is dropped
   from every table. Both models now emit Urdu script directly.
3. **`N_SAMPLES = 150`** - this is the first Omnilingual-vs-Whisper run, so it starts small.

Pipeline:
- **Omnilingual**: audio -> forced `urd_Arab` (direct, no conversion)
- **Whisper**: audio -> forced Urdu (direct, no conversion)
- Both: clean punctuation + izafat -> score. GEC is done manually in Gemini (free).

**Run order:** GPU on -> run to the GEC dump -> correct in Gemini -> load back -> final tables.
> With the converter gone, WER is no longer carrying transliteration noise, so WER and CER
> should now be read together rather than treating CER as the only trustworthy metric.

---

## HOW TO CHANGE THE SAMPLE SIZE

**The `# 2) Imports + config + GPU check` cell, the line that reads:**

```python
SEED, N_SAMPLES = 42, 150
```

Change the `150`. That single number controls both datasets. Nothing else needs editing.

### How large can it go?

| Dataset | Where clips come from | Usable pool |
|---|---|---|
| FLEURS `ur_pk` | test 299 + validation 267 + train 2109 | **2675** |
| UrduSpeech news+podcast (Std) | benchmark 175 + corpus 1558 | **1733** |

At `N_SAMPLES = 150` only the FLEURS `test` split and the UrduSpeech `benchmark` root are
touched, so nothing large is downloaded. Above 566 FLEURS clips you pay for a **1.6 GB**
parquet download; above 175 UrduSpeech clips the loader moves into the corpus root.

### Two Omnilingual-specific constraints

- **Clips longer than 40 s are rejected** by the Omnilingual inference pipeline. The loader
  now drops them up front (see `drop_long`) so both models score the exact same clip set -
  the effective N can therefore come in slightly under 150.
- **VRAM**: `omniASR_LLM_1B_v2` fits a free-Colab T4. `_3B_v2` needs ~10 GB and `_7B_v2`
  ~17 GB, so those require an A100/L4-class GPU.

### Cost warning

Manual GEC volume is still **4 x N lines** (2 datasets x 2 models). At `N_SAMPLES = 150`
that is ~600 lines through Gemini in ~40-line chunks.

***note for users: change runtime type to 2025.10 before running this notebook
because fairseq2 isn't compatible with python 3.13.x***

In [ ]:
!python --version

In [ ]:
# 1) setup

In [ ]:
!apt-get -qq update && apt-get -qq install -y libsndfile1
!pip install -q --force-reinstall torch==2.8.0 torchaudio==2.8.0 torchvision==0.23.0 --index-url https://download.pytorch.org/whl/cu128
!pip install -q "pyarrow==20.0.0"
!pip install -q fairseq2==0.6.0 fairseq2n==0.6.0 --extra-index-url https://fair.pkg.atmeta.com/fairseq2/whl/pt2.8.0/cu128
!pip install -q --ignore-requires-python omnilingual-asr==0.2.0
!pip install -q pandas jiwer datasets==3.2.0 accelerate librosa soundfile huggingface_hub transformers

***restart the runtime once installation completes.***

In [1]:
import torch, fairseq2, omnilingual_asr, datasets, pyarrow
print(torch.__version__, fairseq2.__version__, datasets.__version__, pyarrow.__version__)

2.8.0+cu128 0.6 3.2.0 20.0.0


In [2]:
# 2) Imports + config + GPU check
import re, gc, os, time, unicodedata, string, json
import numpy as np, pandas as pd, torch, jiwer
from tqdm.auto import tqdm
DEVICE="cuda" if torch.cuda.is_available() else "cpu"
DTYPE=torch.float16 if DEVICE=="cuda" else torch.float32
assert DEVICE=="cuda", "GPU is OFF. Turn on the GPU accelerator, then Run all."
print("Device:", DEVICE, "|", torch.cuda.get_device_name(0))
# CHANGED (v3): Qwen3-ASR -> Meta Omnilingual ASR. Model cards are plain strings resolved
# by the omnilingual_asr package, not HF repo ids.
OMNI_MODEL   = "omniASR_LLM_1B_v2"   # fits a T4. Larger: _3B_v2 (~10 GB), _7B_v2 (~17 GB)
OMNI_LANG    = "urd_Arab"            # {language}_{script}; Urdu is natively supported
OMNI_MAX_SEC = 40                    # pipeline hard limit - longer clips are dropped at load time
WHISPER_MODEL="openai/whisper-large-v3"

# >>>>>>>>>>>>>>>>>  SAMPLE SIZE - CHANGE THIS ONE NUMBER  <<<<<<<<<<<<<<<<<
# CHANGED (v3): 1200 -> 150. First Omnilingual-vs-Whisper pass starts small.
SEED, N_SAMPLES = 42, 150      # max usable: FLEURS 2675, UrduSpeech news+podcast 1733
# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>

DATASETS = ["FLEURS", "UrduSpeech_news_podcast"]

# FLEURS splits are consumed in this order; the loader stops once the pool covers
# N_SAMPLES. 'train' is a 1.6 GB download, so it is only touched when you need >566.
FLEURS_SPLITS = ["test", "validation", "train"]

# UrduSpeech: benchmark first (so small N reproduces the original run), then the corpus.
US_REPO   = "ASLP-lab/UrduSpeech"
US_ROOTS  = ["benchmark/US-benchmark-Std/", "corpus/US-Std/"]   # Standard Urdu, no code-switch
US_WANT   = ("news", "podcast")

OUT_DIR = "gec_v2_results"
os.makedirs(OUT_DIR, exist_ok=True)
print(f"N_SAMPLES = {N_SAMPLES}  |  results -> {OUT_DIR}/")

# ---- timing infrastructure -------------------------------------------------
# CHANGED (v3): no transliteration step left to time.
# Every heavy step (data loading, transcription) records one row
# here. Section 13 turns TIMES into timing_breakdown.csv and a printed summary.
_T0 = time.time()
TIMES = []
def record_time(step, seconds, **meta):
    row = {"step": step, "seconds": round(seconds, 3), "minutes": round(seconds / 60, 3)}
    row.update(meta)
    TIMES.append(row)
    extra = " ".join(f"{k}={v}" for k, v in meta.items())
    print(f"[time] {step:<26} {seconds:8.1f}s  ({seconds/60:5.2f} min)  {extra}")
class Timed:
    # with Timed("step_name", dataset="FLEURS", n=150): ...   records on exit.
    def __init__(self, step, **meta): self.step, self.meta = step, meta
    def __enter__(self): self.t0 = time.time(); return self
    def __exit__(self, *exc): record_time(self.step, time.time() - self.t0, **self.meta)

Device: cuda | Tesla T4
N_SAMPLES = 150  |  results -> gec_v2_results/


In [3]:
# 3) Cleaners
# CHANGED (v3): the Devanagari->Urdu converter is gone - Omnilingual outputs Urdu script.
_DIAC_RE = re.compile("[\u0610-\u061a\u0640\u064b-\u0670\u06d6-\u06dc\u06df-\u06e8\u06ea-\u06ed]")
_PUNCT = set(string.punctuation) | set("\u06d4\u060c\u061f\u061b\u066a\u0964\u0965\u201c\u201d"
                                       "\u2018\u2019\u2014\u2013\u2026\u00ab\u00bb\u2016\u2010\u2012\u00b7")
_IZAFAT = {"\u0627\u06d2", "\u0627\u06cc", "\u0626\u06d2", "\u0621", "\u06d2", "\u0627\u0648"}
_UMAP = {"\u064a":"\u06cc", "\u0643":"\u06a9", "\u0629":"\u06c1", "\u06c3":"\u06c1",
         "\u0649":"\u06cc", "\u0625":"\u0627", "\u0623":"\u0627", "\u0622":"\u0627",
         "\u0672":"\u0627", "\u0673":"\u0627", "\u0624":"\u0648", "\u0626":"\u06cc"}
def clean_urdu(t):                     # full clean used for SCORING
    t = unicodedata.normalize("NFC", str(t)); t = _DIAC_RE.sub("", t)
    t = "".join(_UMAP.get(ch, ch) for ch in t)
    t = "".join(" " if ch in _PUNCT else ch for ch in t)
    return " ".join(w for w in t.split() if w not in _IZAFAT).strip()
def strip_diac(t):                     # light clean = readable Urdu for Gemini
    return re.sub(r"\s+", " ", _DIAC_RE.sub("", unicodedata.normalize("NFC", str(t)))).strip()
def wer_cer(refs, hyps):
    r=[clean_urdu(x) for x in refs]; h=[clean_urdu(x) for x in hyps]
    return jiwer.wer(r,h), jiwer.cer(r,h)
def wer_cer_row(ref, hyp):
    # Per-clip score. (None, None) when the reference is empty after cleaning.
    r, h = clean_urdu(ref), clean_urdu(hyp)
    if not r: return (None, None)
    if not h: return (1.0, 1.0)
    return (jiwer.wer(r, h), jiwer.cer(r, h))
print("ready |", clean_urdu("\u0645\u06cc\u06ba \u0622\u062c \u0628\u0627\u0632\u0627\u0631 \u062c\u0627 \u0631\u06c1\u0627 \u06c1\u0648\u06ba\u06d4"))

ready | میں اج بازار جا رہا ہوں


# **WER breakdown function**

In [5]:
def wer_breakdown(refs, hyps):
    r = [clean_urdu(x) for x in refs]
    h = [clean_urdu(x) for x in hyps]
    out = jiwer.process_words(r, h)
    total_ref_words = out.hits + out.substitutions + out.deletions
    return {
        "wer": out.wer,
        "hits": out.hits,
        "substitutions": out.substitutions,
        "deletions": out.deletions,
        "insertions": out.insertions,
        "sub_rate_%": round(100*out.substitutions/total_ref_words, 2),
        "del_rate_%": round(100*out.deletions/total_ref_words, 2),
        "ins_rate_%": round(100*out.insertions/total_ref_words, 2),
    }

In [6]:
# 4) Model runners (loaded one at a time to save GPU memory)
# CHANGED (v3): run_qwen_hindi -> run_omni_urdu. Omnilingual is not a transformers model,
# so AutoProcessor/AutoModelForMultimodalLM are no longer imported; it exposes its own
# ASRInferencePipeline which takes {"waveform", "sample_rate"} dicts plus a language code.
from transformers import pipeline
from omnilingual_asr.models.inference.pipeline import ASRInferencePipeline
def clear_gpu():
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache(); torch.cuda.ipc_collect()
_omni={}
def run_omni_urdu(audio_list, batch_size=2, sr=16000):
    # CHANGED (v3): batch_size 4 -> 2. The LLM-ASR decoder is heavier per clip than Qwen's
    # and 2 keeps a T4 inside its 16 GB with the 1B card. Raise it on a bigger GPU.
    if "p" not in _omni:
        _omni["p"] = ASRInferencePipeline(model_card=OMNI_MODEL, device="cuda", dtype=DTYPE)
        print("Omnilingual loaded:", OMNI_MODEL, "| lang:", OMNI_LANG)
    p = _omni["p"]; hyps = []
    for s in tqdm(range(0, len(audio_list), batch_size), desc="Omni(Urdu)"):
        b = audio_list[s:s+batch_size]
        batch = [{"waveform": a, "sample_rate": sr} for a in b]
        try:
            hyps.extend(p.transcribe(batch, lang=[OMNI_LANG]*len(b), batch_size=len(b)))
        except Exception as e:
            # One bad batch should not cost the whole run; retry clip-by-clip, blank on failure.
            print(f"  batch at {s} failed ({type(e).__name__}); retrying one by one")
            for one in batch:
                try: hyps.extend(p.transcribe([one], lang=[OMNI_LANG], batch_size=1))
                except Exception: hyps.append("")
    return hyps
def free_omni():
    if _omni: _omni.clear(); clear_gpu()

In [7]:
# 5) Loaders for both datasets
from huggingface_hub import list_repo_files, hf_hub_download
from datasets import load_dataset, Audio, concatenate_datasets
import soundfile as sf, librosa
from collections import Counter

def load_fleurs():
    ff = list_repo_files("google/fleurs", repo_type="dataset", revision="refs/convert/parquet")
    parts, pool = [], 0
    for split in FLEURS_SPLITS:
        paths = sorted(f for f in ff
                       if f.startswith("ur_pk/" + split + "/") and f.endswith(".parquet"))
        if not paths:
            continue
        local = [hf_hub_download("google/fleurs", p, repo_type="dataset",
                                 revision="refs/convert/parquet") for p in paths]
        d = load_dataset("parquet", data_files=local, split="train")
        parts.append(d); pool += len(d)
        print(f"  FLEURS {split}: +{len(d)} clips (pool {pool})")
        if pool >= N_SAMPLES:
            break                       # stop before paying for the 1.6 GB train parquet
    ds = parts[0] if len(parts) == 1 else concatenate_datasets(parts)
    n = min(N_SAMPLES, len(ds))
    if n < N_SAMPLES:
        print(f"  !! FLEURS pool is {len(ds)}; using {n} instead of {N_SAMPLES}")
    ds = ds.shuffle(seed=SEED).select(range(n)).cast_column("audio", Audio(sampling_rate=16000))
    return [e["audio"]["array"] for e in ds], [e["transcription"] for e in ds]

# CHANGED (v3): Omnilingual rejects clips longer than OMNI_MAX_SEC (40 s). They are dropped
# at load time rather than at transcription time, so Omnilingual and Whisper are scored on
# exactly the same clip set. Effective N can come in slightly under N_SAMPLES as a result.
def drop_long(audios, golds, max_sec=OMNI_MAX_SEC, sr=16000):
    keep = [i for i, a in enumerate(audios) if len(a) / sr <= max_sec]
    if len(keep) < len(audios):
        print(f"  dropped {len(audios)-len(keep)} clip(s) longer than {max_sec}s")
    return [audios[i] for i in keep], [golds[i] for i in keep]

def _load_wav(path, sr=16000):
    a, s = sf.read(path, dtype="float32")
    if a.ndim > 1: a = a.mean(axis=1)
    if s != sr: a = librosa.resample(a, orig_sr=s, target_sr=sr)
    return a

def load_urduspeech_news_podcast():
    allf = list_repo_files(US_REPO, repo_type="dataset")

    # Transcript files: benchmark roots first, so small N reproduces the original run.
    jsonls = []
    for root in US_ROOTS:
        jsonls += [f for f in allf
                   if f.startswith(root) and f.endswith(".jsonl")
                   and "paralinguistic" not in f.lower()
                   and any(w in f.lower() for w in US_WANT)]
    print("transcript files matched:", len(jsonls))
    for f in jsonls: print("   ", f)

    # Index audio PER CATEGORY DIRECTORY. A repo-wide basename index is unsafe here:
    # 682 basenames occur in more than one subset, so it can pair a transcript with
    # the audio of a different subset.
    audio_by_dir = {}
    for f in allf:
        if f.lower().endswith(".wav"):
            audio_by_dir.setdefault(f.rsplit("/audio/", 1)[0], {})[f.rsplit("/", 1)[-1].lower()] = f

    A, G, CATS = [], [], []
    for jf in jsonls:
        if len(A) >= N_SAMPLES: break
        aidx = audio_by_dir.get(jf.rsplit("/", 1)[0], {})
        if not aidx: continue
        for line in open(hf_hub_download(US_REPO, jf, repo_type="dataset"), encoding="utf-8"):
            line = line.strip()
            if not line: continue
            try: e = json.loads(line)
            except Exception: continue
            text = e.get("ground_truth") or e.get("Transcription")
            name = (e.get("Audio_Clip") or e.get("audio_id") or "").rsplit("/", 1)[-1].lower()
            rp = aidx.get(name)
            if not text or not str(text).strip() or rp is None: continue
            A.append(_load_wav(hf_hub_download(US_REPO, rp, repo_type="dataset")))
            G.append(str(text)); CATS.append(e.get("Audio_category", ""))
            if len(A) >= N_SAMPLES: break
    if len(A) < N_SAMPLES:
        print(f"  !! UrduSpeech pool exhausted at {len(A)} clips (asked for {N_SAMPLES})")
    print("  categories:", Counter(CATS))
    return A, G

LOADERS={"FLEURS":load_fleurs, "UrduSpeech_news_podcast":load_urduspeech_news_podcast}
data={}
for name in DATASETS:
    t0 = time.time()
    a,g=LOADERS[name]()
    a,g=drop_long(a,g)                      # CHANGED (v3): enforce the 40 s Omnilingual limit
    record_time("data_loading", time.time()-t0, dataset=name, n=len(a))
    data[name]=(a,g); print(name,"->",len(a),"clips")

ur_pk/test/0000.parquet: reconstructing file:   0%|          |  0.00B /  188MB            

ur_pk/test/0000.parquet: downloading bytes:           |  0.00B            

Generating train split: 0 examples [00:00, ? examples/s]

  FLEURS test: +299 clips (pool 299)
[time] data_loading                   20.0s  ( 0.33 min)  dataset=FLEURS n=150
FLEURS -> 150 clips
transcript files matched: 9
    benchmark/US-benchmark-Std/long/NEWS/clean_transcription.jsonl
    benchmark/US-benchmark-Std/long/PODCAST/clean_transcription.jsonl
    benchmark/US-benchmark-Std/long/Podcast/clean_transcription.jsonl
    benchmark/US-benchmark-Std/short/NEWS/clean_transcription.jsonl
    benchmark/US-benchmark-Std/short/PODCAST/clean_transcription.jsonl
    benchmark/US-benchmark-Std/short/Podcast/clean_transcription.jsonl
    corpus/US-Std/long/news/news_final_transcription.jsonl
    corpus/US-Std/long/podcast/podcast_final_transcription.jsonl
    corpus/US-Std/short/podcast/podcast_final_transcription.jsonl


clean_transcription.jsonl:   0%|          | 0.00/55.0k [00:00<?, ?B/s]

...

benchmark/US-benchmark-Std/short/Podcast(…): reconstructing file:   0%|          |  0.00B /  593kB            

benchmark/US-benchmark-Std/short/Podcast(…): downloading bytes:           |  0.00B            

benchmark/US-benchmark-Std/short/Podcast(…): reconstructing file:   0%|          |  0.00B /  390kB            

benchmark/US-benchmark-Std/short/Podcast(…): downloading bytes:           |  0.00B            

  categories: Counter({'NEWS': 91, 'Podcast': 55, 'PODCAST': 4})
[time] data_loading                  268.8s  ( 4.48 min)  dataset=UrduSpeech_news_podcast n=150
UrduSpeech_news_podcast -> 150 clips


In [ ]:
#quick omni check
audios, gold = data["FLEURS"]

# Omnilingual, 3 clips
out = run_omni_urdu(audios[:3])
for h, g in zip(out, gold[:3]):
    print("HYP:", h)
    print("REF:", g)
    print()
print(wer_cer(gold[:3], out))

In [8]:
# 6) Phase A: Omnilingual on all datasets (then free it)
# CHANGED (v3): was Qwen + a separate transliteration pass. Omnilingual returns Urdu script
# directly, so there is a single timed step here instead of two, and no `omni_hindi` stage.
results={name:{} for name in DATASETS}
for name in DATASETS:
    audios,gold=data[name]
    results[name]["gold"]=gold
    with Timed("omni_transcription", dataset=name, n=len(audios)):
        results[name]["omni"]=run_omni_urdu(audios)
    print(name,"Omnilingual done")
free_omni()

100%|██████████| 8.49G/8.49G [00:46<00:00, 197MB/s] 


Output()

100%|██████████| 89.3k/89.3k [00:00<00:00, 29.0MB/s]

Omnilingual loaded: omniASR_LLM_1B_v2 | lang: urd_Arab


Omni(Urdu):   0%|          | 0/75 [00:00<?, ?it/s]

[time] omni_transcription            556.7s  ( 9.28 min)  dataset=FLEURS n=150
FLEURS Omnilingual done


Omni(Urdu):   0%|          | 0/75 [00:00<?, ?it/s]

[time] omni_transcription            607.1s  (10.12 min)  dataset=UrduSpeech_news_podcast n=150
UrduSpeech_news_podcast Omnilingual done


In [16]:
import pickle
pickle.dump(results, open("omni_results.pkl", "wb"))

In [18]:
   import transformers.pipelines.automatic_speech_recognition as asr_mod
   asr_mod.is_torchcodec_available = lambda: False
   clear_gpu()

In [20]:
clear_gpu()

In [21]:
#new run_whisper_urdu

from transformers import AutoProcessor, AutoModelForSpeechSeq2Seq

def run_whisper_urdu(audio_list, batch_size=1):
   proc = AutoProcessor.from_pretrained(WHISPER_MODEL)
   model = AutoModelForSpeechSeq2Seq.from_pretrained(
       WHISPER_MODEL, torch_dtype=DTYPE, low_cpu_mem_usage=True).to(DEVICE)
   outs = []
   for a in tqdm(audio_list, desc="Whisper(Urdu)"):
       a = np.asarray(a, dtype=np.float32)
       if len(a) > 30 * 16000:   # long-form: keep the full clip
           inp = proc(a, sampling_rate=16000, return_tensors="pt", truncation=False,
                      padding="longest", return_attention_mask=True)
       else:
           inp = proc(a, sampling_rate=16000, return_tensors="pt")
       inp = inp.to(DEVICE, DTYPE)
       with torch.no_grad():
           ids = model.generate(**inp, language="urdu", task="transcribe",
                                return_timestamps=True)
       outs.append(proc.batch_decode(ids, skip_special_tokens=True)[0])
   del model; clear_gpu()
   return outs

In [22]:
# 7) Phase B: Whisper (direct Urdu) on all datasets
for name in DATASETS:
    audios,_=data[name]
    with Timed("whisper_transcription", dataset=name, n=len(audios)):
        results[name]["whisper"]=run_whisper_urdu(audios)
    print(name,"Whisper done")

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/1259 [00:00<?, ?it/s]

Whisper(Urdu):   0%|          | 0/150 [00:00<?, ?it/s]

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensA

[time] whisper_transcription         382.6s  ( 6.38 min)  dataset=FLEURS n=150
FLEURS Whisper done


Loading weights:   0%|          | 0/1259 [00:00<?, ?it/s]

Whisper(Urdu):   0%|          | 0/150 [00:00<?, ?it/s]

[time] whisper_transcription         441.4s  ( 7.36 min)  dataset=UrduSpeech_news_podcast n=150
UrduSpeech_news_podcast Whisper done


In [23]:
# 8) RAW comparison table (before GEC) - saved immediately, so a later crash costs nothing
rows=[]
for name in DATASETS:
    for model in ["omni","whisper"]:            # CHANGED (v3): "qwen" -> "omni"
        w,c=wer_cer(results[name]["gold"], results[name][model])
        rows.append({"dataset":name,"model":model,"n":len(results[name]["gold"]),
                     "RAW_WER_%":round(w*100,2),"RAW_CER_%":round(c*100,2)})
raw_df=pd.DataFrame(rows)
raw_df.to_csv(f"{OUT_DIR}/raw_scores.csv", index=False, encoding="utf-8-sig")

# CHANGED (v3): the qwen_hindi_deva column is gone - there is no intermediate script now.
pd.DataFrame([{"dataset":name, "n":i+1,
               "gold":results[name]["gold"][i],
               "omni_urdu_raw":results[name]["omni"][i],
               "whisper_urdu_raw":results[name]["whisper"][i]}
              for name in DATASETS for i in range(len(results[name]["gold"]))]
            ).to_csv(f"{OUT_DIR}/raw_transcripts.csv", index=False, encoding="utf-8-sig")

print(raw_df.to_string(index=False))
print(f"\nSaved {OUT_DIR}/raw_scores.csv and {OUT_DIR}/raw_transcripts.csv")

                dataset   model   n  RAW_WER_%  RAW_CER_%
                 FLEURS    omni 150      15.96       5.34
                 FLEURS whisper 150      20.59       7.09
UrduSpeech_news_podcast    omni 150      14.86       7.16
UrduSpeech_news_podcast whisper 150      16.52       7.13

Saved gec_v2_results/raw_scores.csv and gec_v2_results/raw_transcripts.csv


## GEC step (manual, free, in Gemini)

The next cell writes one file per dataset, `gec_input_<dataset>.txt`, holding that dataset's
Omnilingual lines followed by its Whisper lines, numbered `1..2N` **within the dataset**.
Download them, paste into **gemini.google.com** in chunks of ~40 lines with the prompt below,
keep the numbering, then save the corrected text as `gec_FLEURS.txt` and `gec_UrduSpeech.txt`.

> You are correcting a noisy Urdu speech-to-text transcription. For each numbered Urdu line,
> fix the clearly wrong or garbled words so it reads as correct, natural Urdu. Do NOT
> translate, add, delete, or rephrase words, and keep the SAME numbering, one line per input.
> Output only the corrected Urdu.

After each chunk, check Gemini returned the same number of lines before moving on.

In [24]:
# 9) Dump ALL outputs (both models, both datasets) numbered for GEC - one file per dataset
flat=[]   # (dataset, model, local_index, text)
for name in DATASETS:
    for model in ["omni","whisper"]:            # CHANGED (v3): "qwen" -> "omni"
        for i,o in enumerate(results[name][model]):
            flat.append((name,model,i,strip_diac(o)))

PREVIEW = 20
for name in DATASETS:
    lines = [t for (n,_,_,t) in flat if n == name]
    path = f"{OUT_DIR}/gec_input_{name}.txt"
    with open(path,"w",encoding="utf-8") as f:
        for k,t in enumerate(lines,1): f.write(f"{k}. {t}\n")
    print(f"===== {name}: {len(lines)} lines -> {path} =====")
    print("\n".join(f"{k+1}. {t}" for k,t in enumerate(lines[:PREVIEW])))
    if len(lines) > PREVIEW:
        print(f"... ({len(lines)-PREVIEW} more lines in the file)\n")

try:
    from google.colab import files as _f
    for name in DATASETS: _f.download(f"{OUT_DIR}/gec_input_{name}.txt")
except Exception as e:
    print("(not on Colab - take the files from the file browser)", type(e).__name__)

===== FLEURS: 300 lines -> gec_v2_results/gec_input_FLEURS.txt =====
1. وہ بہت سے ممالک کے لئے بینک نوٹ کی کندہ کاری میں بھی مصروف تھا اس کے قائم کے حالیہ مصالحوں میں سے کینیڈا کے نئے 5 ڈالر اور 100 ڈالر کے نوٹوں کے سامنے والے حصے پر وزیر اعظم کی تصویر بنانا شامل ہے
2. احتیاط رکھیں کہ آپکا کپڑا زیادہ گرم نہ ہوپائے ورنہ یا تو وہ سکر جائے گا یا جھلس جائے گا
3. ایک بم گورنر جنرل کے دفتر کے باہر پھٹا
4. یہاں اب بھی بہت سے مرد و خواتین زندہ ہیں جنھوں نے یہاں زندہ رہنے کی جدوجہد کی اور بہت سے لوگ ہیں جن کے اپنوں کا قتل کر دیا گیا یا مرنے تک کام کرایا گیا جیسے یہودی اور غیر یہودی
5. لیٹ وائس لائق نیشنل پارک گھنے جنگلات سے گرا ہے جس میں بنیادی اس طور پر سفید اس نوبر اور چیر کے درخت ہیں اور اس کی خاص بات کوہی اور خشکی سے گری نباتات کا امتزاج ہے
6. یہ شہر افریقہ سے کچھ بھڑکیلے پہاڑی گوریلا کا سوراخ لگانے کے ساتھ نائرا گونگو آتش فشا پر چڑھنے کی چوکی بھی ہے
7. بہت سے لوگ ان کو ڈائنوسور تسلیم نہیں کرتے ہیں کیونکہ وہ بال وپڑ رکھتے ہیں اور اڑنے کے قابل تھے
8. نیکسٹ چین ایک ایسا نظام ہے جو کہ ایف اے ا

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [27]:
# 10) Load the Gemini-corrected files back (one per dataset, numbered 1..2N locally).
# Use the Colab picker, or drop gec_FLEURS.txt / gec_UrduSpeech.txt into the working
# directory and re-run - the cell falls back to reading them from disk.
import glob
up = {os.path.basename(p): open(p, encoding="utf-8").read()
      for p in glob.glob("/kaggle/input/datasets/nukhbamolai/inputt/gec*.txt")}
print("read from disk:", list(up))
assert up, "No GEC files found. Upload them, or put gec_<dataset>.txt in the working directory."

ranges, start = {}, 0
for name in DATASETS:
    count = sum(1 for (n,_,_,_) in flat if n == name)
    ranges[name] = (start, count)
    start += count

def match_dataset(fname):
    fn = fname.lower().replace("_", "").replace(" ", "")
    for name in DATASETS:
        key = name.lower().replace("_", "").split()[0][:10]   # 'fleurs' / 'urduspeech'
        if key in fn:
            return name
    raise ValueError(f"Could not match filename '{fname}' to a dataset name: {DATASETS}")

gec_flat = [strip_diac(t) for (_,_,_,t) in flat]

for fname, raw in up.items():
    if "gec_input" in fname.lower():
        print("skipping the un-corrected dump:", fname); continue
    name = match_dataset(fname)
    lo, count = ranges[name]
    filled = 0
    for line in raw.splitlines():
        m = re.match(r"^\s*(\d+)[\.\)]\s*(.*)$", line.strip())
        if m:
            local_k = int(m.group(1)) - 1
            if 0 <= local_k < count and m.group(2).strip():
                gec_flat[lo + local_k] = m.group(2).strip(); filled += 1
    print(f"{fname} -> {name}: {filled}/{count} lines")

# CHANGED (v3): "qwen" -> "omni"
gec = {name:{mo:[None]*len(results[name][mo]) for mo in ["omni","whisper"]} for name in DATASETS}
for k,(name,model,i,_) in enumerate(flat):
    gec[name][model][i] = gec_flat[k]
print("GEC lines loaded:", sum(1 for x in gec_flat if x))

read from disk: ['gec_UrduSpeech.txt', 'gec_FLEURS.txt']
gec_UrduSpeech.txt -> UrduSpeech_news_podcast: 300/300 lines
gec_FLEURS.txt -> FLEURS: 300/300 lines
GEC lines loaded: 600


In [28]:
# 11) Final comparison table: RAW vs GEC, Omnilingual vs Whisper, both datasets
rows=[]
for name in DATASETS:
    for model in ["omni","whisper"]:            # CHANGED (v3): "qwen" -> "omni"
        rw,rc=wer_cer(results[name]["gold"], results[name][model])
        gw,gc_=wer_cer(results[name]["gold"], gec[name][model])
        rows.append({"dataset":name,"model":model,"n":len(results[name]["gold"]),
                     "RAW_WER_%":round(rw*100,2),"RAW_CER_%":round(rc*100,2),
                     "GEC_WER_%":round(gw*100,2),"GEC_CER_%":round(gc_*100,2),
                     "WER_gain":round((rw-gw)*100,2),"CER_gain":round((rc-gc_)*100,2)})
final=pd.DataFrame(rows)
# CHANGED (v3): output file renamed to match the new model pair.
final.to_csv(f"{OUT_DIR}/omni_vs_whisper_GEC_comparison.csv",index=False,encoding="utf-8-sig")
print(final.to_string(index=False))
print(f"\nSaved {OUT_DIR}/omni_vs_whisper_GEC_comparison.csv")

                dataset   model   n  RAW_WER_%  RAW_CER_%  GEC_WER_%  GEC_CER_%  WER_gain  CER_gain
                 FLEURS    omni 150      15.96       5.34      13.92       5.55      2.04     -0.21
                 FLEURS whisper 150      20.59       7.09      13.73       5.47      6.86      1.61
UrduSpeech_news_podcast    omni 150      14.86       7.16      10.52       6.47      4.33      0.69
UrduSpeech_news_podcast whisper 150      16.52       7.13      11.28       7.09      5.24      0.04

Saved gec_v2_results/omni_vs_whisper_GEC_comparison.csv


# **Breakdown computation**

In [29]:
breakdown_rows = []
for name in DATASETS:
    for model in ["omni", "whisper"]:           # CHANGED (v3): "qwen" -> "omni"
        for stage, hyp_list in [("RAW", results[name][model]), ("GEC", gec[name][model])]:
            b = wer_breakdown(results[name]["gold"], hyp_list)
            breakdown_rows.append({"dataset": name, "model": model, "stage": stage, **b})

breakdown_df = pd.DataFrame(breakdown_rows)
breakdown_df.to_csv(f"{OUT_DIR}/wer_error_breakdown.csv", index=False, encoding="utf-8-sig")
print(breakdown_df.to_string(index=False))

                dataset   model stage      wer  hits  substitutions  deletions  insertions  sub_rate_%  del_rate_%  ins_rate_%
                 FLEURS    omni   RAW 0.159620  3632            509         69          94       12.09        1.64        2.23
                 FLEURS    omni   GEC 0.139192  3732            410         68         108        9.74        1.62        2.57
                 FLEURS whisper   RAW 0.205938  3454            691         65         111       16.41        1.54        2.64
                 FLEURS whisper   GEC 0.137292  3743            401         66         111        9.52        1.57        2.64
UrduSpeech_news_podcast    omni   RAW 0.148566  4381            466        208          77        9.22        4.11        1.52
UrduSpeech_news_podcast    omni   GEC 0.105242  4601            254        200          78        5.02        3.96        1.54
UrduSpeech_news_podcast whisper   RAW 0.165183  4287            633        135          67       12.52        2

## Save the samples

`samples_<dataset>.csv` is the sheet to read: one row per clip, gold next to all four
hypotheses, with per-row WER and CER for each. `samples_all.csv` is the same concatenated
across datasets; `per_clip_scores.csv` is the long/tidy form for plotting.

In [30]:
# 12) Clean side-by-side sample sheets + per-clip scores
# CHANGED (v3): the two Qwen arms become Omnilingual arms; column names follow.
ARMS = [("omni",        "omni_urdu_raw",     lambda n,i: results[n]["omni"][i]),
        ("omni_gec",    "omni_urdu_gec",     lambda n,i: gec[n]["omni"][i]),
        ("whisper",     "whisper_urdu_raw",  lambda n,i: results[n]["whisper"][i]),
        ("whisper_gec", "whisper_urdu_gec",  lambda n,i: gec[n]["whisper"][i])]

wide_all, per_clip = [], []
for name in DATASETS:
    gold = results[name]["gold"]
    rows = []
    for i in range(len(gold)):
        # CHANGED (v3): qwen_hindi_deva column removed (no intermediate script).
        r = {"dataset": name, "n": i+1,
             "gold": gold[i],
             "gold_clean": clean_urdu(gold[i])}
        for arm, col, get in ARMS:
            hyp = get(name, i)
            r[col] = hyp
            r[col + "_clean"] = clean_urdu(hyp)
            w, c = wer_cer_row(gold[i], hyp)
            r[arm + "_WER"] = None if w is None else round(w, 4)
            r[arm + "_CER"] = None if c is None else round(c, 4)
            per_clip.append({"dataset": name, "n": i+1, "arm": arm,
                             "WER": None if w is None else round(w, 4),
                             "CER": None if c is None else round(c, 4)})
        rows.append(r)
    df = pd.DataFrame(rows)
    df.to_csv(f"{OUT_DIR}/samples_{name}.csv", index=False, encoding="utf-8-sig")
    wide_all.append(df)
    print(f"{name}: {len(df)} rows -> {OUT_DIR}/samples_{name}.csv")

samples_all = pd.concat(wide_all, ignore_index=True)
samples_all.to_csv(f"{OUT_DIR}/samples_all.csv", index=False, encoding="utf-8-sig")
pd.DataFrame(per_clip).to_csv(f"{OUT_DIR}/per_clip_scores.csv", index=False, encoding="utf-8-sig")

json.dump({"n_samples_requested": N_SAMPLES, "seed": SEED, "datasets": DATASETS,
           "clips_loaded": {n: len(results[n]["gold"]) for n in DATASETS},
           "omni": OMNI_MODEL, "omni_lang": OMNI_LANG,      # CHANGED (v3)
           "omni_max_sec": OMNI_MAX_SEC, "whisper": WHISPER_MODEL,
           "fleurs_splits": FLEURS_SPLITS, "urduspeech_roots": US_ROOTS},
          open(f"{OUT_DIR}/run_config.json", "w"), indent=2)

print(f"\nFiles in {OUT_DIR}/:")
for f in sorted(os.listdir(OUT_DIR)):
    print(f"  {f:<38} {os.path.getsize(os.path.join(OUT_DIR,f))/1024:9.1f} KB")

FLEURS: 150 rows -> gec_v2_results/samples_FLEURS.csv
UrduSpeech_news_podcast: 150 rows -> gec_v2_results/samples_UrduSpeech_news_podcast.csv

Files in gec_v2_results/:
  gec_input_FLEURS.txt                        68.2 KB
  gec_input_UrduSpeech_news_podcast.txt       78.7 KB
  omni_vs_whisper_GEC_comparison.csv           0.3 KB
  per_clip_scores.csv                         46.7 KB
  raw_scores.csv                               0.2 KB
  raw_transcripts.csv                        223.8 KB
  run_config.json                              0.5 KB
  samples_FLEURS.csv                         344.1 KB
  samples_UrduSpeech_news_podcast.csv        404.8 KB
  samples_all.csv                            748.7 KB
  wer_error_breakdown.csv                      0.7 KB


## Timing breakdown

`timing_breakdown.csv` has one row per (step, dataset): data loading, Omnilingual
transcription, Whisper transcription - each with seconds, minutes and seconds-per-clip.
Two summary rows are appended: the **sum of every timed step** (the actual compute cost) and
the **wall-clock time since Section 2** (which also includes any pause you took to correct
text in Gemini, since that step is not itself timed). The gap between the two tells you how
long the manual GEC pass took.

> CHANGED (v3): the `deva_to_urdu_transliteration` row is gone - that stage no longer exists,
> so the whole pipeline cost is now GPU time only.

In [31]:
# 13) Timing summary: per step, per dataset, and overall
timing_df = pd.DataFrame(TIMES)
if not timing_df.empty and "n" in timing_df.columns:
    timing_df["sec_per_item"] = (timing_df["seconds"] / timing_df["n"]).round(3)
compute_total = float(timing_df["seconds"].sum()) if not timing_df.empty else 0.0
wall_total = time.time() - _T0

rows = timing_df.to_dict("records") if not timing_df.empty else []
rows.append({"step": "OVERALL_COMPUTE_sum_of_timed_steps", "seconds": round(compute_total, 3),
            "minutes": round(compute_total/60, 3)})
rows.append({"step": "OVERALL_WALL_CLOCK_incl_manual_GEC_wait", "seconds": round(wall_total, 3),
            "minutes": round(wall_total/60, 3)})
timing_full = pd.DataFrame(rows)
timing_full.to_csv(f"{OUT_DIR}/timing_breakdown.csv", index=False, encoding="utf-8-sig")

print("=" * 92)
print("TIMING BREAKDOWN")
print("=" * 92)
cols = [c for c in ["step", "dataset", "n", "seconds", "minutes", "sec_per_item"]
        if c in timing_full.columns]
print(timing_full[cols].to_string(index=False))

if not timing_df.empty and "dataset" in timing_df.columns:
    print("\nMinutes by step x dataset:")
    piv = timing_df.pivot_table(index="step", columns="dataset", values="minutes", aggfunc="sum")
    piv["TOTAL"] = piv.sum(axis=1)
    print(piv.round(2).to_string())

print(f"\nSaved {OUT_DIR}/timing_breakdown.csv")
print(f"Compute time (loading + transcription + transliteration only): {compute_total/60:.1f} min")
print(f"Wall-clock time so far (includes any manual GEC pause):        {wall_total/60:.1f} min")

TIMING BREAKDOWN
                                   step                 dataset     n  seconds  minutes  sec_per_item
                           data_loading                  FLEURS 150.0   19.978    0.333         0.133
                           data_loading UrduSpeech_news_podcast 150.0  268.827    4.480         1.792
                     omni_transcription                  FLEURS 150.0  556.664    9.278         3.711
                     omni_transcription UrduSpeech_news_podcast 150.0  607.119   10.119         4.047
                  whisper_transcription                  FLEURS 150.0   26.108    0.435         0.174
                  whisper_transcription                  FLEURS 150.0    5.314    0.089         0.035
                  whisper_transcription                  FLEURS 150.0  382.557    6.376         2.550
                  whisper_transcription UrduSpeech_news_podcast 150.0  441.437    7.357         2.943
     OVERALL_COMPUTE_sum_of_timed_steps                     NaN  

In [ ]:
# 14) Quick eyeball: first 3 clips of each dataset, rendered right-to-left
from IPython.display import HTML, display
_S = ("font-family:'Noto Nastaliq Urdu','Jameel Noori Nastaleeq','Noto Naskh Arabic',serif;"
      "font-size:17px;line-height:2.2;direction:rtl;text-align:right;unicode-bidi:isolate;"
      "padding:2px 8px;background:#fafafa;border-right:3px solid #ddd;")
_L = "direction:ltr;text-align:left;font-family:monospace;font-size:11px;color:#777;margin-top:6px;"
for name in DATASETS:
    for i in range(min(3, len(results[name]["gold"]))):
        html = f"<div style='direction:ltr;font-family:monospace;font-weight:bold'>{name} #{i+1}</div>"
        for lab, txt in [("GOLD",          results[name]["gold"][i]),
                         ("OMNI raw",      results[name]["omni"][i]),    # CHANGED (v3)
                         ("OMNI + GEC",    gec[name]["omni"][i]),
                         ("WHISPER raw",   results[name]["whisper"][i]),
                         ("WHISPER + GEC", gec[name]["whisper"][i])]:
            html += f"<div style='{_L}'>{lab}</div><div style='{_S}'>{txt}</div>"
        display(HTML(html + "<hr style='border:none;border-top:1px solid #eee'>"))

In [ ]:
# 15) Download everything (Colab only; harmless elsewhere)
try:
    from google.colab import files
    for f in ["omni_vs_whisper_GEC_comparison.csv", "wer_error_breakdown.csv",   # CHANGED (v3)
              "samples_all.csv", "per_clip_scores.csv", "timing_breakdown.csv"]:
        files.download(f"{OUT_DIR}/{f}")
except Exception as e:
    print("not on Colab:", type(e).__name__, "- take the files from", OUT_DIR)